<a href="https://colab.research.google.com/github/ch-jsk/jsk-flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ch-jsk/jsk-flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I use the anonymized starter dataset, which contains one row per content item and trailing-90-day metrics. The target is whether the observed trend is declining, derived from `trend_direction == "down"`. The feature vector uses candidate performance, position, content-age, search-demand, and content-structure fields that are available in the dataset.

I exclude the trend fields and identifiers from model inputs. Missing numeric values are imputed using the training-set median, and missing categorical values receive an explicit missing category. Where source fields are missing, I create missingness indicators rather than silently treating missing values as zero.

This is a feature-vector and leakage audit, not evidence that the selected features are causal or that the resulting model is ready for deployment.

In [2]:
!git clone https://github.com/ch-jsk/jsk-flyrank-ml-internship.git

Cloning into 'jsk-flyrank-ml-internship'...
remote: Enumerating objects: 214, done.
remote: Counting objects: 100% (214/214), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 214 (delta 102), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (214/214), 2.14 MiB | 11.34 MiB/s, done.
Resolving deltas: 100% (102/102), done.


In [8]:

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Load the smaller dataset if it is not already loaded.
if "df" not in globals() or not isinstance(df, pd.DataFrame):
    from pathlib import Path

    csv_path = Path(
        "/content/jsk-flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
    )

    if not csv_path.exists():
        csv_path = Path("data/raw/content_refresh_anonymized.csv")

    if not csv_path.exists():
        raise FileNotFoundError(
            "CSV not found. Run the repository setup cell first."
        )

    df = pd.read_csv(csv_path)

assert "trend_direction" in df.columns, (
    "trend_direction is missing. Check the actual dataset columns."
)

# Target: 1 = declining, 0 = not declining.
y = (
    df["trend_direction"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("down")
    .astype("int8")
)
y.name = "target_declining"

numeric_candidates = [
    "impressions_90d",
    "avg_position",
    "content_age_days",
    "engagement_rate",
    "search_volume",
    "word_count",
]

categorical_candidates = ["content_type", "main_intent"]

numeric_features = [c for c in numeric_candidates if c in df.columns]
categorical_features = [c for c in categorical_candidates if c in df.columns]

X = df[numeric_features + categorical_features].copy()

# Zero position means no position data, not rank zero.
if "avg_position" in X.columns:
    X["avg_position_missing_or_no_data"] = (
        X["avg_position"].isna() | X["avg_position"].eq(0)
    ).astype("int8")
    X.loc[X["avg_position"].eq(0), "avg_position"] = np.nan

# Convert numeric candidates safely and add missingness indicators.
for col in numeric_features:
    X[col] = pd.to_numeric(X[col], errors="coerce")
    X[f"{col}_is_missing"] = X[col].isna().astype("int8")

numeric_model_features = [
    c for c in X.columns
    if pd.api.types.is_numeric_dtype(X[c])
]
categorical_model_features = [
    c for c in X.columns
    if c not in numeric_model_features
]

transformers = []

if numeric_model_features:
    transformers.append((
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        numeric_model_features,
    ))

if categorical_model_features:
    transformers.append((
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(
                strategy="constant", fill_value="__MISSING__"
            )),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]),
        categorical_model_features,
    ))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)

print("Dataset:", df.shape)
print("Feature matrix:", X.shape)
print("Features:", X.columns.tolist())
print("Target created:", y.name)
print("Target rows:", len(y))


Dataset: (30000, 44)
Feature matrix: (30000, 15)
Features: ['impressions_90d', 'avg_position', 'content_age_days', 'engagement_rate', 'search_volume', 'word_count', 'content_type', 'main_intent', 'avg_position_missing_or_no_data', 'impressions_90d_is_missing', 'avg_position_is_missing', 'content_age_days_is_missing', 'engagement_rate_is_missing', 'search_volume_is_missing', 'word_count_is_missing']
Target created: target_declining
Target rows: 30000


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature group | Meaning | Missing-value handling | Available at prediction time? |
|---|---|---|---|
| `impressions_90d` | Search visibility over the trailing 90-day window | Median imputation plus missingness indicator | Only if the window ends before the prediction time |
| `avg_position` | Average search position; zero means no data | Zero converted to missing, then median imputation; explicit no-data flag | Only if calculated from a completed historical window |
| `content_age_days` | Age of the content | Median imputation plus missingness indicator | Yes, if calculated as of the prediction date |
| `engagement_rate` | Observed engagement rate | Median imputation plus missingness indicator | Only if measured before the prediction time |
| `search_volume` | Search-demand estimate | Median imputation plus missingness indicator | Only if the estimate was available at prediction time |
| `word_count` | Content length proxy | Median imputation plus missingness indicator | Yes, if taken from the current content snapshot |
| `content_type`, `main_intent` | Content category or intent, when present | Explicit missing category and one-hot encoding | Only if known at prediction time |
| Missingness flags | Whether a candidate input is missing or unavailable | Binary indicator | Yes, if based only on the same historical inputs |

The prediction-time statements above are conditional on the source fields being captured before the prediction cutoff. The CSV contains aggregated trailing-90-day data, so its export alone does not prove when every field became available. `trend_direction` and `trend_pct` are excluded because they are used to define or describe the target, while `client_id` and `content_id` are identifiers reserved for grouping and joins.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_notes = pd.DataFrame([
    {
        "feature": col,
        "meaning": (
            "Numeric candidate input or missingness indicator"
            if col in numeric_model_features
            else "Categorical candidate input"
        ),
        "missing_handling": (
            "Median imputation; missingness flags added where applicable"
            if col in numeric_model_features
            else "Explicit missing category; one-hot encoding"
        ),
        "prediction_time_check": "Verify source was available before prediction cutoff",
    }
    for col in X.columns
])

display(feature_notes)

print("Features used:", len(X.columns))
print("Identifier columns in X:", [
    c for c in X.columns
    if c.lower() in {"client_id", "content_id", "url", "url_hash_id"}
])
print("Known label-derived columns in X:", [
    c for c in X.columns
    if c.lower() in {"trend_direction", "trend_pct", "is_declining_label"}
])


,feature,meaning,missing_handling,prediction_time_check
0,impressions_90d,Numeric candidate input or missingness indicator,Median imputation; missingness flags added whe...,Verify source was available before prediction ...
1,avg_position,Numeric candidate input or missingness indicator,Median imputation; missingness flags added whe...,Verify source was available before prediction ...
2,content_age_days,Numeric candidate input or missingness indicator,Median imputation; missingness flags added whe...,Verify source was available before prediction ...
3,engagement_rate,Numeric candidate input or missingness indicator,Median imputation; missingness flags added whe...,Verify source was available before prediction ...
4,search_volume,Numeric candidate input or missingness indicator,Median imputation; missingness flags added whe...,Verify source was available before prediction ...
5,word_count,Numeric candidate input or missingness indicator,Median imputation; missingness flags added whe...,Verify source was available before prediction ...
6,content_type,Categorical candidate input,Explicit missing category; one-hot encoding,Verify source was available before prediction ...
7,main_intent,Categorical candidate input,Explicit missing category; one-hot encoding,Verify source was available before prediction ...
8,avg_position_missing_or_no_data,Numeric candidate input or missingness indicator,Median imputation; missingness flags added whe...,Verify source was available before prediction ...
9,impressions_90d_is_missing,Numeric candidate input or missingness indicator,Median imputation; missingness flags added whe...,Verify source was available before prediction ...


Features used: 15
Identifier columns in X: []
Known label-derived columns in X: []


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the candidate feature vector for direct label leakage, identifiers, and columns whose names suggest that they contain future outcomes or precomputed decisions. The target is derived from `trend_direction`, and `trend_pct` contributes to the trend definition; both are excluded from the features.

I also check whether the feature matrix contains duplicate column names or any forbidden fields. A name-based scan is only a first pass: it cannot prove that a feature is leakage-free. The timing and construction of each field must also be verified against the data contract. Because the starter dataset contains a single trailing-90-day snapshot, I cannot establish a strict historical prediction cutoff from this file alone.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Check whether the feature matrix contains forbidden fields.
forbidden_exact = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "target",
    "client_id",
    "content_id",
    "url",
    "url_hash_id",
}

present_forbidden = [
    c for c in X.columns
    if c.lower() in forbidden_exact
]

# 2. Scan the source schema for suspicious names.
suspicious_terms = (
    "trend", "declin", "label", "target", "outcome",
    "future", "next_", "after_", "recommendation", "priority_score"
)

suspicious_source_columns = [
    c for c in df.columns
    if any(term in c.lower() for term in suspicious_terms)
]

# 3. Check identifiers and duplicate feature names.
identifier_columns = [
    c for c in X.columns
    if c.lower() in {"client_id", "content_id", "url", "url_hash_id"}
]

duplicate_feature_names = X.columns[X.columns.duplicated()].tolist()

# 4. Verify that the target is aligned with the source label.
target_matches_source = (
    y.eq(
        df["trend_direction"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("down")
        .astype("int8")
    ).all()
)

print("Forbidden fields found in X:", present_forbidden)
print("Identifier fields found in X:", identifier_columns)
print("Duplicate feature names:", duplicate_feature_names)
print("Target matches documented trend rule:", target_matches_source)

print("\nSource columns requiring manual leakage review:")
print(suspicious_source_columns)

assert not present_forbidden, "Remove forbidden fields from X."
assert not identifier_columns, "Identifiers must not be model features."
assert not duplicate_feature_names, "Resolve duplicate feature names."
assert target_matches_source, "Target construction does not match source label."

print("\nLeakage checks passed. Manual timing review is still required.")


Forbidden fields found in X: []
Identifier fields found in X: []
Duplicate feature names: []
Target matches documented trend rule: True

Source columns requiring manual leakage review:
['trend_direction', 'trend_pct']

Leakage checks passed. Manual timing review is still required.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field | Reason |
|---|---|
| `trend_direction` | Direct source for the target; using it would reveal the answer. |
| `trend_pct` | Used to define or describe the trend outcome; creates direct or indirect leakage risk. |
| `is_declining_label` | Explicit label field, if present in another version of the dataset. |
| `client_id` | Pseudonymous client identifier; reserved for grouped validation, not prediction. |
| `content_id` | Pseudonymous content identifier; reserved for joins and traceability, not prediction. |
| URL and URL-hash fields | Identifiers can reveal identity or memorized associations and are unnecessary for this feature vector. |
| Precomputed priority or recommendation fields | Excluded unless their construction and availability are independently verified; they may encode an earlier decision or the target itself. |
| Future-window metrics | Excluded because they would not be available at the time of a genuine prediction. |

The exclusion policy is intentionally conservative. A field is not safe merely because its name looks harmless: its definition, source window, and availability at prediction time must be checked. This notebook does not claim that every possible leakage pathway has been eliminated.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Explicit exclusion register for this audit.
exclusion_register = pd.DataFrame([
    ("trend_direction", "Defines the target; direct label leakage."),
    ("trend_pct", "Trend outcome/proxy; leakage risk."),
    ("is_declining_label", "Explicit label, if present."),
    ("client_id", "Grouping and validation only; not a feature."),
    ("content_id", "Join/traceability only; not a feature."),
    ("URL or URL-hash fields", "Identifier/privacy and memorization risk."),
    ("Precomputed priority/recommendation fields", "Potential target proxy; requires provenance review."),
    ("Future-window metrics", "Unavailable at the prediction cutoff."),
], columns=["excluded_field", "reason"])

display(exclusion_register)

# Show which registered exclusions actually exist in this source.
print("\nPresence in the current source:")
for field, reason in exclusion_register.itertuples(index=False, name=None):
    if field in {"URL or URL-hash fields"}:
        present = [
            c for c in df.columns
            if "url" in c.lower() or "url_hash" in c.lower()
        ]
    elif field == "Precomputed priority/recommendation fields":
        present = [
            c for c in df.columns
            if "priority" in c.lower() or "recommendation" in c.lower()
        ]
    elif field == "Future-window metrics":
        present = [
            c for c in df.columns
            if "future" in c.lower() or "next_" in c.lower()
        ]
    else:
        present = [field] if field in df.columns else []

    print(f"{field}: {present if present else 'not found by this exact-name check'}")

print("\nFinal feature count:", X.shape[1])
print("Rows:", X.shape[0])
print("Target positive rate:", round(float(y.mean()), 4))


,excluded_field,reason
0,trend_direction,Defines the target; direct label leakage.
1,trend_pct,Trend outcome/proxy; leakage risk.
2,is_declining_label,"Explicit label, if present."
3,client_id,Grouping and validation only; not a feature.
4,content_id,Join/traceability only; not a feature.
5,URL or URL-hash fields,Identifier/privacy and memorization risk.
6,Precomputed priority/recommendation fields,Potential target proxy; requires provenance re...
7,Future-window metrics,Unavailable at the prediction cutoff.



Presence in the current source:
trend_direction: ['trend_direction']
trend_pct: ['trend_pct']
is_declining_label: not found by this exact-name check
client_id: ['client_id']
content_id: ['content_id']
URL or URL-hash fields: not found by this exact-name check
Precomputed priority/recommendation fields: not found by this exact-name check
Future-window metrics: not found by this exact-name check

Final feature count: 15
Rows: 30000
Target positive rate: 0.5421


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.